In [1]:
from pathlib import Path
import sys
import inspect

import pandas as pd
import numpy as np

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features import (
    to_monthly_prices,
    compute_momentum,
    compute_moving_average,
    compute_trend_filter,
)

from src.signals import (
    compute_eligibility,
    rank_by_momentum,
    select_top_n_weights,
    lag_weights_for_execution,
    build_momentum_weights,
)

from src.backtest import backtest_monthly_allocation

In [2]:
functions = [
    to_monthly_prices,
    compute_momentum,
    compute_moving_average,
    compute_trend_filter,
    compute_eligibility,
    rank_by_momentum,
    select_top_n_weights,
    lag_weights_for_execution,
    backtest_monthly_allocation,
]

for function in functions:
    print(function.__name__, inspect.signature(function))

to_monthly_prices (prices: 'pd.Series | pd.DataFrame') -> 'pd.Series | pd.DataFrame'
compute_momentum (monthly_prices: 'pd.Series | pd.DataFrame', lookback_months: 'int' = 12, skip_months: 'int' = 1) -> 'pd.Series | pd.DataFrame'
compute_moving_average (monthly_prices: 'pd.Series | pd.DataFrame', window: 'int' = 10) -> 'pd.Series | pd.DataFrame'
compute_trend_filter (monthly_prices: 'pd.Series | pd.DataFrame', moving_average: 'pd.Series | pd.DataFrame') -> 'pd.Series | pd.DataFrame'
compute_eligibility (momentum: 'pd.DataFrame', trend_filter: 'pd.DataFrame', require_positive_momentum: 'bool' = False) -> 'pd.DataFrame'
rank_by_momentum (momentum: 'pd.DataFrame', eligible: 'pd.DataFrame') -> 'pd.DataFrame'
select_top_n_weights (momentum: 'pd.DataFrame', eligible: 'pd.DataFrame', top_n: 'int' = 1, cash_col: 'str' = 'CASH') -> 'pd.DataFrame'
lag_weights_for_execution (target_weights: 'pd.DataFrame', periods: 'int' = 1, cash_col: 'str' = 'CASH') -> 'pd.DataFrame'
backtest_monthly_allocation

In [3]:
dates = pd.to_datetime([
    "2024-01-02",
    "2024-01-15",
    "2024-01-31",
    "2024-02-05",
    "2024-02-20",
    "2024-02-29",
])

daily_prices = pd.Series(
    [100, 103, 105, 104, 108, 110],
    index=dates,
    name="ETF_TEST",
)

monthly_prices = to_monthly_prices(daily_prices)
monthly_prices

2024-01-31    105
2024-02-29    110
Freq: ME, Name: ETF_TEST, dtype: int64

In [4]:
assert monthly_prices.iloc[0] == 105
assert monthly_prices.iloc[1] == 110
assert isinstance(monthly_prices.index, pd.DatetimeIndex)

print("to_monthly_prices validée.")

to_monthly_prices validée.


In [5]:
dates = pd.date_range(
    "2024-01-31",
    periods=4,
    freq="ME",
)

prices = pd.Series(
    [100.0, 110.0, 121.0, 133.1],
    index=dates,
    name="ETF_TEST",
)

momentum_2_0 = compute_momentum(
    prices,
    lookback_months=2,
    skip_months=0,
)

momentum_2_0

2024-01-31     NaN
2024-02-29     NaN
2024-03-31    0.21
2024-04-30    0.21
Freq: ME, Name: ETF_TEST, dtype: float64

In [7]:
assert np.isclose(momentum_2_0.iloc[2], 0.21)

In [8]:
momentum_3_1 = compute_momentum(
    prices,
    lookback_months=3,
    skip_months=1,
)

momentum_3_1

2024-01-31     NaN
2024-02-29     NaN
2024-03-31     NaN
2024-04-30    0.21
Freq: ME, Name: ETF_TEST, dtype: float64

In [9]:
assert np.isclose(momentum_3_1.iloc[3], 0.21)

print("compute_momentum validée.")

compute_momentum validée.


In [10]:
prices = pd.Series(
    [100.0, 110.0, 120.0, 90.0],
    index=pd.date_range("2024-01-31", periods=4, freq="ME"),
)

moving_average = compute_moving_average(
    prices,
    window=3,
)

moving_average

2024-01-31           NaN
2024-02-29           NaN
2024-03-31    110.000000
2024-04-30    106.666667
Freq: ME, dtype: float64

In [11]:
assert pd.isna(moving_average.iloc[0])
assert pd.isna(moving_average.iloc[1])
assert np.isclose(moving_average.iloc[2], 110.0)

print("compute_moving_average validée.")

compute_moving_average validée.


In [12]:
prices = pd.Series(
    [100.0, 110.0, 90.0],
    index=pd.date_range("2024-01-31", periods=3, freq="ME"),
)

moving_average = pd.Series(
    [95.0, 105.0, 100.0],
    index=prices.index,
)

trend = compute_trend_filter(
    prices,
    moving_average,
)

trend

2024-01-31     True
2024-02-29     True
2024-03-31    False
Freq: ME, dtype: bool

In [13]:
expected = pd.Series(
    [True, True, False],
    index=prices.index,
)

pd.testing.assert_series_equal(trend, expected)

print("compute_trend_filter validée.")

compute_trend_filter validée.


In [14]:
dates = pd.date_range(
    "2024-01-31",
    periods=4,
    freq="ME",
)

momentum = pd.DataFrame(
    {
        "ETF_A": [0.10, 0.15, 0.05, 0.20],
        "ETF_B": [0.05, 0.20, 0.10, 0.12],
    },
    index=dates,
)

trend_filter = pd.DataFrame(
    {
        "ETF_A": [True, True, False, True],
        "ETF_B": [True, False, True, True],
    },
    index=dates,
)

In [15]:
eligibility = compute_eligibility(
    momentum,
    trend_filter,
)

eligibility

,ETF_A,ETF_B
2024-01-31,True,True
2024-02-29,True,False
2024-03-31,False,True
2024-04-30,True,True


In [16]:
ranking = rank_by_momentum(
    momentum,
    eligibility,
)

ranking

,ETF_A,ETF_B
2024-01-31,1.0,2.0
2024-02-29,1.0,NaN
2024-03-31,NaN,1.0
2024-04-30,1.0,2.0


In [17]:
target_weights = select_top_n_weights(
    momentum,
    eligibility,
    top_n=1,
)

target_weights

,ETF_A,ETF_B,CASH
2024-01-31,1.0,0.0,0.0
2024-02-29,1.0,0.0,0.0
2024-03-31,0.0,1.0,0.0
2024-04-30,1.0,0.0,0.0


In [18]:
assert (target_weights >= 0).all().all()
assert np.allclose(target_weights.sum(axis=1), 1.0)

In [19]:
executed_weights = lag_weights_for_execution(
    target_weights,
    periods=1,
)

comparison = pd.concat(
    {
        "Target": target_weights,
        "Executed": executed_weights,
    },
    axis=1,
)

comparison

Target            Executed           
            ETF_A ETF_B CASH    ETF_A ETF_B CASH
2024-01-31    1.0   0.0  0.0      0.0   0.0  1.0
2024-02-29    1.0   0.0  0.0      1.0   0.0  0.0
2024-03-31    0.0   1.0  0.0      1.0   0.0  0.0
2024-04-30    1.0   0.0  0.0      0.0   1.0  0.0

In [20]:
pd.testing.assert_series_equal(
    executed_weights.iloc[1],
    target_weights.iloc[0],
    check_names=False,
)

In [21]:
assert np.isclose(
    executed_weights.iloc[0]["CASH"],
    1.0,
)

print("Décalage anti-look-ahead validé.")

Décalage anti-look-ahead validé.


In [22]:
dates = pd.date_range(
    "2024-01-31",
    periods=4,
    freq="ME",
)

prices = pd.DataFrame(
    {
        "ETF_A": [100.0, 110.0, 121.0, 133.1],
        "ETF_B": [100.0, 100.0, 100.0, 100.0],
    },
    index=dates,
)

In [23]:
weights = pd.DataFrame(
    {
        "ETF_A": [0.0, 1.0, 1.0, 1.0],
        "ETF_B": [0.0, 0.0, 0.0, 0.0],
        "CASH":  [1.0, 0.0, 0.0, 0.0],
    },
    index=dates,
)

In [24]:
result = backtest_monthly_allocation(
    monthly_prices=prices,
    executed_weights=weights,
    initial_capital=10_000,
    transaction_cost=0.0,
)

result

,Gross Return,Turnover,Transaction Cost,Net Return,Equity Curve,Invested Exposure,Cash Weight,Selected Assets
2024-01-31,0.0,0.0,0.0,0.0,10000.0,0.0,1.0,CASH
2024-02-29,0.1,1.0,0.0,0.1,11000.0,1.0,0.0,ETF_A
2024-03-31,0.1,0.0,0.0,0.1,12100.0,1.0,0.0,ETF_A
2024-04-30,0.1,0.0,0.0,0.1,13310.0,1.0,0.0,ETF_A


In [25]:
assert not result.empty
assert result.isna().sum().sum() == 0

In [26]:
result.columns.tolist()

['Gross Return',
 'Turnover',
 'Transaction Cost',
 'Net Return',
 'Equity Curve',
 'Invested Exposure',
 'Cash Weight',
 'Selected Assets']

In [27]:
result.tail()

,Gross Return,Turnover,Transaction Cost,Net Return,Equity Curve,Invested Exposure,Cash Weight,Selected Assets
2024-01-31,0.0,0.0,0.0,0.0,10000.0,0.0,1.0,CASH
2024-02-29,0.1,1.0,0.0,0.1,11000.0,1.0,0.0,ETF_A
2024-03-31,0.1,0.0,0.0,0.1,12100.0,1.0,0.0,ETF_A
2024-04-30,0.1,0.0,0.0,0.1,13310.0,1.0,0.0,ETF_A


In [28]:
result_without_costs = backtest_monthly_allocation(
    monthly_prices=prices,
    executed_weights=weights,
    initial_capital=10_000,
    transaction_cost=0.0,
)

result_with_costs = backtest_monthly_allocation(
    monthly_prices=prices,
    executed_weights=weights,
    initial_capital=10_000,
    transaction_cost=0.003,
)

In [29]:
result_without_costs.columns

Index(['Gross Return', 'Turnover', 'Transaction Cost', 'Net Return',
       'Equity Curve', 'Invested Exposure', 'Cash Weight', 'Selected Assets'],
      dtype='object')

In [30]:
assert (
    result_with_costs["Equity Curve"].iloc[-1]
    <= result_without_costs["Equity Curve"].iloc[-1]
)

print("Impact des frais validé.")

Impact des frais validé.
